<h1 align="center" style="color:blue">
    <b>
        Initial Settings
    </b>
</h1>

In [6]:
import glob
import os
import rasterio
from rasterio.plot import show
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import assets.jalali as jalali

%config Completer.use_jedi = False
%matplotlib inline

<h1 align="center" style="color:blue">
    <b>
        Land Based Datasets
    </b>
</h1>

<h3 align="left" style="color:green">
    <b>
        Load Dataset
    </b>
</h3>

In [ ]:
Land_Based_Datasets = pd.read_csv(filepath_or_buffer="Land_Based_Datasets/Dataset/Land_Based_Datasets.csv")

Land_Based_Datasets

In [ ]:
geeID = pd.read_csv("Land_Based_Datasets/GIS/Land_Based_Stations.csv")

geeID

<h3 align="left" style="color:green">
    <b>
        Data Cleansing
    </b>
</h3>

In [ ]:
Land_Based_Datasets.Date_Miladi = pd.to_datetime(arg=Land_Based_Datasets.Date_Miladi,
                                                 format='%Y-%m-%d')

Land_Based_Datasets["Year_Shamsi"] = Land_Based_Datasets["Date_Shamsi"].apply(lambda x : x.split("-")[0])
Land_Based_Datasets["Month_Shamsi"] = Land_Based_Datasets["Date_Shamsi"].apply(lambda x : x.split("-")[1])


LBD_Precip = Land_Based_Datasets.groupby([Land_Based_Datasets['Station'].rename("Station"),
                                           Land_Based_Datasets['Year_Shamsi'].rename("Year"),
                                           Land_Based_Datasets['Month_Shamsi'].rename("Month")])['Precip'].sum().reset_index()

LBD_Precip.sort_values(by=["Station", "Year", "Month"], inplace=True)

LBD_Precip.columns = ["Station", "Year", "Month", "Land_Based_Precip"]

LBD_Precip['Year'] = LBD_Precip['Year'].astype(int)
LBD_Precip['Month'] = LBD_Precip['Month'].astype(int)

LBD_Precip

<h1 align="center" style="color:blue">
    <b>
        ERA5 Daily Aggregates Dataset
    </b>
</h1>

<h3 align="left" style="color:green">
    <b>
        Load Dataset
    </b>
</h3>

In [ ]:
ERA5_Daily_Aggregates_Precip = pd.read_csv("ERA5_Daily_Aggregates_Dataset/Dataset/ERA5_Daily_Aggregates_Precip.csv")

ERA5_Daily_Aggregates_Precip

<h3 align="left" style="color:green">
    <b>
        Data Cleansing
    </b>
</h3>

In [ ]:
ERA5_Daily_Aggregates_Precip["Date_Miladi"] = ERA5_Daily_Aggregates_Precip.startDate.str.split("T", 1, expand=True)[0]
ERA5_Daily_Aggregates_Precip["Date_Miladi"] = pd.to_datetime(arg=ERA5_Daily_Aggregates_Precip["Date_Miladi"], format='%Y-%m-%d')

ERA5_Daily_Aggregates_Precip["Date_Shamsi"] = list(map(lambda i : jalali.Gregorian(i).persian_string(),
                                                       ERA5_Daily_Aggregates_Precip["Date_Miladi"].astype(str)))

ERA5_Daily_Aggregates_Precip["Year_Shamsi"] = ERA5_Daily_Aggregates_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[0].astype(int)
ERA5_Daily_Aggregates_Precip["Month_Shamsi"] = ERA5_Daily_Aggregates_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[1].astype(int)
ERA5_Daily_Aggregates_Precip["Day_Shamsi"] = ERA5_Daily_Aggregates_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[2].astype(int)

ERA5_Daily_Aggregates_Precip = ERA5_Daily_Aggregates_Precip.groupby([ERA5_Daily_Aggregates_Precip['geeID'].rename("geeID"),
                                                                     ERA5_Daily_Aggregates_Precip['Year_Shamsi'].rename("Year"),
                                                                     ERA5_Daily_Aggregates_Precip['Month_Shamsi'].rename("Month")])['mean'].sum().reset_index().rename(columns={"mean" : "ERA5_Precip"})

ERA5_Daily_Aggregates_Precip["ERA5_Precip"] = ERA5_Daily_Aggregates_Precip["ERA5_Precip"] * 1000

ERA5_Daily_Aggregates_Precip = ERA5_Daily_Aggregates_Precip.query('Year >= 1358 and Year <= 1398')

ERA5_Daily_Aggregates_Precip = pd.merge(left=ERA5_Daily_Aggregates_Precip,
                                        right=geeID,
                                        how='left',
                                        on=['geeID'])[['Station', 'Year', 'Month', 'ERA5_Precip']]

ERA5_Daily_Aggregates_Precip.sort_values(by=["Station", "Year", "Month"], inplace=True)

ERA5_Daily_Aggregates_Precip['Year'] = ERA5_Daily_Aggregates_Precip['Year'].astype(int)
ERA5_Daily_Aggregates_Precip['Month'] = ERA5_Daily_Aggregates_Precip['Month'].astype(int)


ERA5_Daily_Aggregates_Precip

<h1 align="center" style="color:blue">
    <b>
        ERA5 Land Hourly Dataset
    </b>
</h1>

<h3 align="left" style="color:green">
    <b>
        Load Dataset
    </b>
</h3>

In [ ]:
ERA5_Land_Precip = pd.read_csv("C:/Users/POOYA/Google Drive/GEE/ECMWF_ERA5_LAND_HOURLY_total_precipitation_1981_2020.csv")

ERA5_Land_Precip

<h3 align="left" style="color:green">
    <b>
        Data Cleansing
    </b>
</h3>

In [ ]:
ERA5_Land_Precip["Date_Miladi"] = ERA5_Land_Precip.startDate.str.split("T", 1, expand=True)[0]
ERA5_Land_Precip["Date_Miladi"] = pd.to_datetime(arg=ERA5_Land_Precip["Date_Miladi"], format='%Y-%m-%d')

ERA5_Land_Precip["Date_Shamsi"] = list(map(lambda i : jalali.Gregorian(i).persian_string(),
                                           ERA5_Land_Precip["Date_Miladi"].astype(str)))

ERA5_Land_Precip["Year_Shamsi"] = ERA5_Land_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[0].astype(int)
ERA5_Land_Precip["Month_Shamsi"] = ERA5_Land_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[1].astype(int)
ERA5_Land_Precip["Day_Shamsi"] = ERA5_Land_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[2].astype(int)

ERA5_Land_Precip = ERA5_Land_Precip.groupby([ERA5_Land_Precip['geeID'].rename("geeID"),
                                             ERA5_Land_Precip['Year_Shamsi'].rename("Year"),
                                             ERA5_Land_Precip['Month_Shamsi'].rename("Month")])['mean'].sum().reset_index().rename(columns={"mean" : "ERA5_Land_Precip"})

ERA5_Land_Precip["ERA5_Land_Precip"] = ERA5_Land_Precip["ERA5_Land_Precip"] * 1000 / 10

ERA5_Land_Precip = ERA5_Land_Precip.query('Year >= 1358 and Year <= 1398')

ERA5_Land_Precip = pd.merge(left=ERA5_Land_Precip,
                            right=geeID,
                            how='left',
                            on=['geeID'])[['Station', 'Year', 'Month', 'ERA5_Land_Precip']]

ERA5_Land_Precip.sort_values(by=["Station", "Year", "Month"], inplace=True)

ERA5_Land_Precip['Year'] = ERA5_Land_Precip['Year'].astype(int)
ERA5_Land_Precip['Month'] = ERA5_Land_Precip['Month'].astype(int)

ERA5_Land_Precip

<h1 align="center" style="color:blue">
    <b>
        WaPOR v2 Datasets
    </b>
</h1>

<h3 align="left" style="color:green">
    <b>
        Load Dataset
    </b>
</h3>

In [ ]:
Land_Based_PointData = gpd.read_file(filename="Land_Based_Datasets/GIS/Land_Based_Stations.shp")

WaPOR_Dataset_Folder = "L1_PCP_D"

WaPOR_Dataset_Files = [os.path.basename(x) for x in glob.glob(pathname="WaPOR_v2_Datasets/Dataset/" + WaPOR_Dataset_Folder + "/*.tif")]

fig, ax = plt.subplots(figsize=(12,12))
Land_Based_PointData.plot(ax=ax, color='red')
show(rasterio.open("WaPOR_v2_Datasets/Dataset/" + WaPOR_Dataset_Folder + "/" + WaPOR_Dataset_Files[0]), ax=ax)

<h3 align="left" style="color:green">
    <b>
        Data Cleansing
    </b>
</h3>

In [ ]:
WaPOR_Precip = pd.DataFrame(columns=["geeID", "name", "parameter", "x", "y", "z", "year", "month", "day", "dekadal", "value"])

level = WaPOR_Dataset_Folder.split(sep="_")[0]
parameter = WaPOR_Dataset_Folder.split(sep="_")[1]
timeStep = WaPOR_Dataset_Folder.split(sep="_")[2]

for raster in WaPOR_Dataset_Files:
    if timeStep == "D":
        year = raster.replace(WaPOR_Dataset_Folder,"").replace("\\","").replace("_WAPOR_DEKAD_","").replace(".tif", "").split(sep=".")[0]
        month = raster.replace(WaPOR_Dataset_Folder,"").replace("\\","").replace("_WAPOR_DEKAD_","").replace(".tif", "").split(sep=".")[1]
        dekadal  = raster.replace(WaPOR_Dataset_Folder,"").replace("\\","").replace("_WAPOR_DEKAD_","").replace(".tif", "").split(sep=".")[2]
        if dekadal == "01":
            dekadal = "01"
            day = "05"  
        if dekadal == "11":
            dekadal = "02"
            day = "15"
        if dekadal == "21":
            dekadal = "03"
            day = "25"
            
    rasterData = rasterio.open("WaPOR_v2_Datasets/Dataset/" + WaPOR_Dataset_Folder + "/" + raster)
    
    for point in Land_Based_PointData.iterrows():
        x = point[1]["geometry"].xy[0][0]
        y = point[1]["geometry"].xy[1][0]
        z = point[1]["ELEV"]
        row, col = rasterData.index(x, y)
        value = rasterData.read(1)[row, col]
        ID = point[1]["geeID"]
        name = point[1]["NAME"]        
        
        df = pd.DataFrame([[ID, name, parameter, x, y, z, year, month, day, dekadal, value]],
                          columns=["geeID", "name", "parameter", "x", "y", "z", "year", "month", "day", "dekadal", "value"])
        
        WaPOR_Precip = WaPOR_Precip.append(df, ignore_index=True)
    
WaPOR_Precip["Date_Miladi"] = WaPOR_Precip["year"] + "-" + WaPOR_Precip["month"] + "-" + WaPOR_Precip["day"]
WaPOR_Precip["Date_Miladi"] = pd.to_datetime(arg=WaPOR_Precip["Date_Miladi"], format='%Y-%m-%d')

WaPOR_Precip["Date_Shamsi"] = list(map(lambda i : jalali.Gregorian(i).persian_string(),
                                       WaPOR_Precip["Date_Miladi"].astype(str)))

WaPOR_Precip["Year_Shamsi"] = WaPOR_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[0].astype(int)
WaPOR_Precip["Month_Shamsi"] = WaPOR_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[1].astype(int)
WaPOR_Precip["Day_Shamsi"] = WaPOR_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[2].astype(int)

WaPOR_Precip = WaPOR_Precip.groupby([WaPOR_Precip['name'].rename("Station"),
                                     WaPOR_Precip['Year_Shamsi'].rename("Year"),
                                     WaPOR_Precip['Month_Shamsi'].rename("Month")])['value'].sum().reset_index().rename(columns={"value" : "WaPOR_Precip"})

WaPOR_Precip = WaPOR_Precip.query('Year >= 1388 and Year <= 1398')

WaPOR_Precip["WaPOR_Precip"] = WaPOR_Precip["WaPOR_Precip"]

WaPOR_Precip.sort_values(by=["Station", "Year", "Month"], inplace=True)

WaPOR_Precip['Year'] = WaPOR_Precip['Year'].astype(int)
WaPOR_Precip['Month'] = WaPOR_Precip['Month'].astype(int)

WaPOR_Precip

<h1 align="center" style="color:blue">
    <b>
        PERSIANN CDR Dataset
    </b>
</h1>

<h3 align="left" style="color:green">
    <b>
        Load Dataset
    </b>
</h3>

In [ ]:
PERSIANN_CDR_Precip = pd.read_csv("C:/Users/POOYA/Dropbox/Work/Hydrotech/SABA_Khozestan/Precipitation_Datasets_Analysis/PERSIANN_CDR_Dataset/Dataset/PERSIANN_CDR_Precip.csv")

PERSIANN_CDR_Precip

<h3 align="left" style="color:green">
    <b>
        Data Cleansing
    </b>
</h3>

In [ ]:
PERSIANN_CDR_Precip["Date_Miladi"] = PERSIANN_CDR_Precip.startDate.str.split("T", 1, expand=True)[0]
PERSIANN_CDR_Precip["Date_Miladi"] = pd.to_datetime(arg=PERSIANN_CDR_Precip["Date_Miladi"], format='%Y-%m-%d')

PERSIANN_CDR_Precip["Date_Shamsi"] = list(map(lambda i : jalali.Gregorian(i).persian_string(),
                                              PERSIANN_CDR_Precip["Date_Miladi"].astype(str)))

PERSIANN_CDR_Precip["Year_Shamsi"] = PERSIANN_CDR_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[0].astype(int)
PERSIANN_CDR_Precip["Month_Shamsi"] = PERSIANN_CDR_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[1].astype(int)
PERSIANN_CDR_Precip["Day_Shamsi"] = PERSIANN_CDR_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[2].astype(int)

PERSIANN_CDR_Precip = PERSIANN_CDR_Precip.groupby([PERSIANN_CDR_Precip['geeID'].rename("geeID"),
                                                   PERSIANN_CDR_Precip['Year_Shamsi'].rename("Year"),
                                                   PERSIANN_CDR_Precip['Month_Shamsi'].rename("Month")])['mean'].sum().reset_index().rename(columns={"mean" : "PERSIANN_CDR_Precip"})

PERSIANN_CDR_Precip = PERSIANN_CDR_Precip.query('Year >= 1358 and Year <= 1398')

PERSIANN_CDR_Precip = pd.merge(left=PERSIANN_CDR_Precip,
                               right=geeID,
                               how='left',
                               on=['geeID'])[['Station', 'Year', 'Month', 'PERSIANN_CDR_Precip']]

PERSIANN_CDR_Precip.sort_values(by=["Station", "Year", "Month"], inplace=True)

PERSIANN_CDR_Precip['Year'] = PERSIANN_CDR_Precip['Year'].astype(int)
PERSIANN_CDR_Precip['Month'] = PERSIANN_CDR_Precip['Month'].astype(int)

PERSIANN_CDR_Precip

<h1 align="center" style="color:blue">
    <b>
        CHIRPS Dataset
    </b>
</h1>

<h3 align="left" style="color:green">
    <b>
        Load Dataset
    </b>
</h3>

In [ ]:
CHIRPS_Precip = pd.read_csv("C:/Users/POOYA/Dropbox/Work/Hydrotech/SABA_Khozestan/Precipitation_Datasets_Analysis/CHIRPS_Daily_Dataset/Dataset/CHIRPS_Daily_Precip.csv")

CHIRPS_Precip

<h3 align="left" style="color:green">
    <b>
        Data Cleansing
    </b>
</h3>

In [ ]:
CHIRPS_Precip["Date_Miladi"] = CHIRPS_Precip.startDate.str.split("T", 1, expand=True)[0]
CHIRPS_Precip["Date_Miladi"] = pd.to_datetime(arg=CHIRPS_Precip["Date_Miladi"], format='%Y-%m-%d')

CHIRPS_Precip["Date_Shamsi"] = list(map(lambda i : jalali.Gregorian(i).persian_string(),
                                        CHIRPS_Precip["Date_Miladi"].astype(str)))

CHIRPS_Precip["Year_Shamsi"] = CHIRPS_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[0].astype(int)
CHIRPS_Precip["Month_Shamsi"] = CHIRPS_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[1].astype(int)
CHIRPS_Precip["Day_Shamsi"] = CHIRPS_Precip["Date_Shamsi"].str.split("-", 2, expand=True)[2].astype(int)

CHIRPS_Precip = CHIRPS_Precip.groupby([CHIRPS_Precip['geeID'].rename("geeID"),
                                       CHIRPS_Precip['Year_Shamsi'].rename("Year"),
                                       CHIRPS_Precip['Month_Shamsi'].rename("Month")])['mean'].sum().reset_index().rename(columns={"mean" : "CHIRPS_Precip"})

CHIRPS_Precip = CHIRPS_Precip.query('Year >= 1358 and Year <= 1398')

CHIRPS_Precip = pd.merge(left=CHIRPS_Precip,
                         right=geeID,
                         how='left',
                         on=['geeID'])[['Station', 'Year', 'Month', 'CHIRPS_Precip']]

CHIRPS_Precip.sort_values(by=["Station", "Year", "Month"], inplace=True)

CHIRPS_Precip['Year'] = CHIRPS_Precip['Year'].astype(int)
CHIRPS_Precip['Month'] = CHIRPS_Precip['Month'].astype(int)

CHIRPS_Precip

<h1 align="center" style="color:blue">
    <b>
        Combine Data
    </b>
</h1>

In [ ]:
LBD_Precip = LBD_Precip.reset_index(drop=True)
ERA5_Daily_Aggregates_Precip = ERA5_Daily_Aggregates_Precip.reset_index(drop=True)
ERA5_Land_Precip = ERA5_Land_Precip.reset_index(drop=True)
WaPOR_Precip = WaPOR_Precip.reset_index(drop=True)
PERSIANN_CDR_Precip = PERSIANN_CDR_Precip.reset_index(drop=True)
CHIRPS_Precip = CHIRPS_Precip.reset_index(drop=True)

Precip_Data = pd.merge(left=LBD_Precip,
                       right=ERA5_Daily_Aggregates_Precip,
                       how='outer',
                       on=['Station', 'Year', 'Month']).merge(right=ERA5_Land_Precip,
                                                              how='outer',
                                                              on=['Station', 'Year', 'Month']).merge(right=WaPOR_Precip,
                                                                                                     how='outer',
                                                                                                     on=['Station', 'Year', 'Month'])\
.merge(right=PERSIANN_CDR_Precip,
       how='outer',
       on=['Station', 'Year', 'Month']).merge(right=CHIRPS_Precip,
                                              how='outer',
                                              on=['Station', 'Year', 'Month'])

Precip_Data.sort_values(by=["Station", "Year", "Month"], inplace=True)

Precip_Data

<h1 align="center" style="color:blue">
    <b>
        Save Data
    </b>
</h1>

<h3 align="left" style="color:green">
    <b>
        Monthly Data
    </b>
</h3>

In [ ]:
Precip_Data.to_csv(path_or_buf="Result/Precip_Data_Monthly.csv", mode="w", index=False)

<h3 align="left" style="color:green">
    <b>
        Yearly Data
    </b>
</h3>

In [ ]:
Precip_Data_Yearly = Precip_Data.groupby(["Station", "Year"]).agg(
    Land_Based_Precip = pd.NamedAgg(column="Land_Based_Precip", aggfunc="sum"),
    ERA5_Precip = pd.NamedAgg(column="ERA5_Precip", aggfunc="sum"),
    ERA5_Land_Precip = pd.NamedAgg(column="ERA5_Land_Precip", aggfunc="sum"),
    WaPOR_Precip = pd.NamedAgg(column="WaPOR_Precip", aggfunc="sum"),
    PERSIANN_CDR_Precip = pd.NamedAgg(column="PERSIANN_CDR_Precip", aggfunc="sum"),
    CHIRPS_Precip = pd.NamedAgg(column="CHIRPS_Precip", aggfunc="sum")).reset_index()

Precip_Data_Yearly.replace(0, np.nan, inplace=True)

Precip_Data_Yearly.to_csv(path_or_buf="Result/Precip_Data_Yearly.csv", mode="w", index=False)

<h1 align="center" style="color:blue">
    <b>
        Plot Data
    </b>
</h1>

<h3 align="left" style="color:green">
    <b>
        Data Preparation
    </b>
</h3>

In [7]:
Precip_Data_Yearly = pd.read_csv("Result/Precip_Data_Yearly.csv")

Precip_Data_Plot = pd.melt(frame=Precip_Data_Yearly,
                           id_vars=['Station','Year'],
                           value_vars=['Land_Based_Precip','ERA5_Precip','ERA5_Land_Precip', 'WaPOR_Precip', 'PERSIANN_CDR_Precip', 'CHIRPS_Precip'],
                           var_name='Dataset',
                           value_name='Precip')

Precip_Data_Plot = Precip_Data_Plot.query('Year >= 1380 and Year <= 1394')

Precip_Data_Plot

,Station,Year,Dataset,Precip
22,ABADAN,1380,Land_Based_Precip,188.430000
23,ABADAN,1381,Land_Based_Precip,102.840000
24,ABADAN,1382,Land_Based_Precip,204.030000
25,ABADAN,1383,Land_Based_Precip,160.560000
26,ABADAN,1384,Land_Based_Precip,231.760000
...,...,...,...,...
11553,TANG PANJ - BAKHTIYARI,1390,CHIRPS_Precip,327.582290
11554,TANG PANJ - BAKHTIYARI,1391,CHIRPS_Precip,566.750768
11555,TANG PANJ - BAKHTIYARI,1392,CHIRPS_Precip,557.842481
11556,TANG PANJ - BAKHTIYARI,1393,CHIRPS_Precip,309.708653


<h3 align="left" style="color:green">
    <b>
        Plot
    </b>
</h3>

In [8]:
import dash
import dash_core_components as dcc
import dash_html_components as html
from jupyter_dash import JupyterDash
from dash.dependencies import Input, Output, State
import plotly.graph_objects as go
import plotly.express as px


external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']

# app = JupyterDash(__name__, external_stylesheets=external_stylesheets)
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)

No_Matching_Data_Found_Fig = {
    "layout": {
        "xaxis": {"visible": False},
        "yaxis": {"visible": False},
        "annotations": [
            {
                "text": "NO MATCHING DATA FOUND",
                "xref": "paper",
                "yref": "paper",
                "showarrow": False,
                "font": {"size": 24}
            }
        ]
    }
}

app.layout = html.Div([
    
    html.Div([

        html.Div([
            html.H5("Select A Station:"),
            dcc.Dropdown(
                id="select_station",
                options=[{'label': i, 'value': i} for i in Precip_Data_Plot["Station"].unique()],
                placeholder= "Select A Station"
            ),
        ], style={'width': '25%',
                  'display': 'inline-block'}
        )
    ], style={'border': 'thin lightgrey solid',
              'backgroundColor': 'rgb(250, 250, 250)',
              'padding': '10px 5px'}
    ),

    html.Div([
        dcc.Graph(
            id='time_serise_plot'
        )
    ], style={'width': '100%',
              'display': 'inline-block',
              'padding': '0 20'}
    )
])


@app.callback(Output('time_serise_plot', 'figure'),
              [Input('select_station', 'value')])
def time_serise_plot_update(select_station):
    
    if select_station is None:
        return No_Matching_Data_Found_Fig
    
    df = Precip_Data_Plot[Precip_Data_Plot["Station"] == select_station]
    
#     fig = go.Figure()    
#     fig.add_trace(go.Scatter(x=Precip_Data_Plot.Year,
#                              y=Precip_Data_Plot.Precip,
#                              color=Precip_Data_Plot.Dataset,
#                              mode='lines+markers',
#                              name='lines+markers'))

    fig = px.line(data_frame=df, 
                  x="Year",
                  y="Precip",
                  color='Dataset')
    
    return fig  
    


app.run_server()

 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [21/Feb/2021 10:47:36] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:47:40] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:47:40] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:47:41] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:47:52] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:48:41] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:48:58] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:49:07] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:50:30] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:50:57] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:51:04] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2021 10:51:13] "POST /_dash-update-component

# TerraClimate Data

# Create Error Raster